# Notebook para la predicción de métricas de desempeño generales

## Importación de librerías necesarias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

In [3]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('predict_flow')
spark = spark_utils.spark

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13883999-aebe-4018-9e36-8c5c665143d4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 137ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [4]:
import pyspark.sql.functions as F

## Importar información de referencia

In [7]:
REGENERATE_ITERMEDIATE_TABLES = False

In [8]:
GOLD_SCHEMA_ENCODING = 'gold.encoding'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
GOLD_PREMODELING = 'gold.premodeling'

In [9]:
meta_items_features_text_clean_embeddings = (
    spark.read.format('parquet').load(spark_utils.path(
        'meta_items_features_text_clean_embeddings', 
        catalog = GOLD_SCHEMA_ENCODING
    ))
)

cluster_lsh_candidates = spark.read.format('delta').load(spark_utils.path(
    'lsh_pairs_lsh_pca_features_balanced_pca_features_1000_300', catalog=GOLD_SCHEMA_CLUSTER
))

meta_items_features_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_text_clean', catalog = GOLD_PREMODELING
))

df_items_sample_balanced = spark.read.format('delta').load(spark_utils.path(
    'df_items_sample_balanced', catalog = GOLD_PREMODELING
))

In [8]:
chosen_features = meta_items_features_text_clean_embeddings.alias('A').join(
    meta_items_features_text_clean.alias('B'),
    F.col('A.record_id') == F.col('B.record_id'),
    how = 'inner'
).join(
    df_items_sample_balanced.alias('C'),
    F.col('B.parent_asin') == F.col('C.parent_asin'),
    how = 'inner'
).select(
    F.col('C.parent_asin').alias('parent_asin'),
    'A.*'
)

In [10]:
if REGENERATE_ITERMEDIATE_TABLES:
    chosen_features.write.format('delta').mode('overwrite').save(spark_utils.path(
        'metrics_jtbd_chosen_features', catalog = GOLD_PREMODELING
    ))
chosen_features = spark.read .format('delta').load(spark_utils.path(
    'metrics_jtbd_chosen_features', catalog = GOLD_PREMODELING
))

In [15]:
from src.utils.models.unsupervised.JTBDBased import JTBDBased
jtbd = JTBDBased(spark_utils=spark_utils)
jtbd.calculate_preds_metrics(
    chosen_features, cluster_lsh_candidates
)
jtbd_comparable_metrics = jtbd.metrics_by_product_actual

In [11]:
if REGENERATE_ITERMEDIATE_TABLES:
    jtbd_comparable_metrics.write.format('delta').mode('overwrite').save(spark_utils.path(
        'jtbd_comparable_metrics', catalog = GOLD_PREMODELING
    ))
jtbd_comparable_metrics = spark.read.format('delta').load(spark_utils.path(
    'jtbd_comparable_metrics', catalog = GOLD_PREMODELING
))

In [13]:
jtbd_comparable_metrics.show(10)

+----------------+------------------+------------------+
|parent_asin_base|   score_predicted|     rating_actual|
+----------------+------------------+------------------+
|      B005KVEPVO|0.7749325266372084|0.8846153846153847|
|      B005LIG4RE|0.5838123415015667|               0.8|
|      B005LLYBYE|0.7508603410032039|0.4666666666666667|
|      B005LLYDLA|0.7508603410032039|0.5272727272727272|
|      B005MX9PMO|0.8970812031440116|0.9083076923076924|
|      B005N8W1Q0|0.8265082328883283|0.7770833333333333|
|      B005NAQ4PM|0.8164271059538207|               0.9|
|      B005NZF8ZY|0.7865667548586297|               0.8|
|      B005OOKO3K|0.8971729243628669|              0.72|
|      B005P11X6O|0.5999999999999982|0.7102702702702703|
+----------------+------------------+------------------+
only showing top 10 rows



In [15]:
df_mae = jtbd_comparable_metrics.agg(F.avg(F.abs(F.col("score_predicted") - F.col("rating_actual"))).alias("mae"))
mae = df_mae.collect()[0]["mae"]
print("Global MAE:", mae)

Global MAE: 0.1359192511944922


In [17]:
from pyspark.sql import functions as F

df_mse = jtbd_comparable_metrics.agg(
    F.avg( (F.col("score_predicted") - F.col("rating_actual"))**2 ).alias("mse")
)

mse = df_mse.collect()[0]["mse"]
print("Global MSE:", mse)


Global MSE: 0.0317319185083751


In [18]:
df_rmse = jtbd_comparable_metrics.agg(
    F.sqrt(F.avg( (F.col("score_predicted") - F.col("rating_actual"))**2 )).alias("rmse")
)

rmse = df_rmse.collect()[0]["rmse"]
print("Global RMSE:", rmse)

Global RMSE: 0.17813455169723558


In [12]:
jtbd_comparable_metrics.count()

16475

In [16]:
from pyspark.sql import functions as F

eps = 1e-12

df_kl = jtbd_comparable_metrics.withColumn(
    "p", F.when(F.col("score_predicted") < eps, eps)
         .when(F.col("score_predicted") > 1-eps, 1-eps)
         .otherwise(F.col("score_predicted"))
).withColumn(
    "q", F.when(F.col("rating_actual") < eps, eps)
         .when(F.col("rating_actual") > 1-eps, 1-eps)
         .otherwise(F.col("rating_actual"))
).withColumn(
    "kl_row",
    F.col("q") * F.log(F.col("q") / F.col("p")) +
    (1 - F.col("q")) * F.log((1 - F.col("q")) / (1 - F.col("p")))
)

global_kl = df_kl.agg(F.avg("kl_row").alias("kl_divergence")).collect()[0]["kl_divergence"]
print("Global KL divergence:", global_kl)


Global KL divergence: 0.39497715102640035
